# Self-repellent chemotaxis

Packages used:

In [ ]:
using Pkg
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using StatsBase

## Model definition

The following code generates the model, with the rules for the agents, and defines the medium.

In [ ]:
self_negative = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64, 
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :active => Bool,

        :S => Float64,

        :methyl => Float64,
        :Yp => Float64, 
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :M => Float64
        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, 
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, 
        :Ki => Float64,
        :Ka => Float64,
        :τm => Float64, 
        :α => Float64,  
        :ωFrec => Float64,    
        :Ky => Float64,    
        :Z => Float64,         
        :Kz => Float64,         
        :Yy => Float64,        

        :DMedium => Float64,
        :delta => Float64
    ),

    medium = Dict(
        :mm => Float64
    ),

    agentODE = quote

        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]


        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)
        

        if x < xmin
            idx = Int(floor(Int, (x+(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        elseif x > xmax
            idx = Int(floor(Int, (x-(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        end

        if y < ymin
            idy = Int(floor(Int,(y+ymax-ymin)/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        elseif y > ymax
            idy = Int(floor(Int,(y-(ymax-ymin))/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)
    
        end
        
        mmb = max(0, mm[idx,idy])

        F = ε0 + ε1 * methyl - Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) 
  
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)       

        mx = (ε0 - Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))     

        dt(x) = vx  
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)       
        
    end,

    agentRule = quote
        
            xmin, xmax = simBox[1,1], simBox[1,2]
            ymin, ymax = simBox[2,1], simBox[2,2]

            idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
            idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2) 

            v_run = v
            v_tumble = 0.25 

            speed = active ? v_run : v_tumble

            Dr_tumble = 6.2      
            Dr_total = active ? Dr_run : Dr_tumble

            mm[idx,idy] += S

            if active 
                λ = ωFrec*exp(-G)
                P = 1 - exp(-λ * dt)
                
            else
                λ = ωFrec*exp(G)
                P = 1 - exp(-λ * dt)  
                
            end


            if active 
                λrt = ωFrec*exp(-G) 

                P_rt = 1 - exp(-λrt * dt)
                P = rand() 
                                                   
                if P < P_rt       
                    active = false
                    vx = speed* cos(theta) 
                    vy = speed* sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()       
                else   
                    active = true
                    vx = speed * cos(theta) 
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()      
                end

            else
                λtr = ωFrec*exp(G) 
                P_tr = 1 - exp(-λtr * dt)
                P = rand()

                if P < P_tr
                    active = true
                    vx = speed * cos(theta) 
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                else
                    active = false
                    vx = speed* cos(theta) 
                    vy = speed* sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                end
            end

            
            xmin, xmax = simBox[1,1], simBox[1,2]
            ymin, ymax = simBox[2,1], simBox[2,2]

            # ================
            #  Boundry conditions
            # ================

            if x < xmin
                x += (xmax - xmin)
            elseif x > xmax
                x -= (xmax - xmin)
            end

            if y < ymin
                y += (ymax - ymin)
            elseif y > ymax
                y -= (ymax - ymin)
            end


    end,


    mediumODE = quote
        if @mediumInside()
            dt(mm) = DMedium *(@∂2(1, mm)+ @∂2(2, mm)) - delta*mm
        elseif @mediumBorder(1,-1)
            mm = mm[NMedium[1] - 1, i2_]
        elseif @mediumBorder(1,1)
            mm = mm[1, i2_]
        elseif @mediumBorder(2,1)
            mm = mm[i1_, 1]
        elseif @mediumBorder(2,-1)
            mm = mm[i1_, NMedium[2] - 1]
        end
    end,


    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Heun(),
    neighborsAlg=CBMNeighbors.CellLinked(cellEdge=10)
)

Next, load the community with the parameters set wanted and evolve it.

In [ ]:
com = Community(
    self_negative,
    N=500,
    dt=0.01,
    simBox = [-100.0 150.0; -100.0 150.0],
    NMedium = [250, 250]
)

m = 1/100
g = 1/10000
d = 1

com.Dr_run = 0.062

com.v = 10.0


com.DMedium = 20
com.delta = 0.005 

com.ωFrec = 1.3
com.Ki = 0.0182
com.Ka = 3.0
com.Nrec = 6.0
com.ε0   = 6.0
com.ε1   = -1.0
com.ε2   = 80
com.ε3   = 80

com.τm = 1.0

com.α   = 6.0

com.K = 2.0 

com.Ky = 100.0
com.Kz = 10.0
com.Z = 5.0
com.Yy = 0.1

com.m = 1.        
com.d = 1.        
com.l = 3;

com.x = 0
com.y = 0

com.theta = rand(Uniform(0,2π),com.N)

com.methyl .= 0.0
com.Yp .= com.K

com.active .= true 

com.S = 0.0025

outfile = "cluster_negative.jld2"
steps = 25000

loadToPlatform!(com, preallocateAgents=500)
com.mm = zeros(Float64, com.NMedium...)

jldopen(outfile, "w") do file

    for step in 1:steps
        step!(com)
        if step % 100 == 0
            stepname = @sprintf("step_%06d", step)
            g = JLD2.Group(file, stepname)

            # Agent-level arrays (length = N)
            g["x"] = copy(com.x)
            g["y"] = copy(com.y)
            g["theta"] = copy(com.theta)
            g["mm_grid"] = copy(com.mm)
        end
    end
end

## Analysis

The follwoing code calculates the average spatial distribution of the agents in the given box over a set time.

In [ ]:
filename1 = "cluster_negative.jld2"

function spatial_entropy(x, y, i; nbins=500)
    ax = Axis(fig[1,i], title = "X")
    ax2 = Axis(fig[2,i], title = "Y")

    xedges = range(-100, 150, length=25)
    yedges = range(-100, 150, length=25)

    h = fit(Histogram, (x,y), (xedges, yedges))

    p = h.weights
 
    p = p ./ sum(p)
    
    p = p[p .> 0]

    return -sum(p .* log.(p))
end

function mean_entropy(filename, i;
                      start_step=5000,
                      end_step=25000,
                      save_interval=100)

    entropies = Float64[]

    jldopen(filename, "r") do file

        for step in start_step:save_interval:end_step

            key = @sprintf("step_%06d", step)

            if haskey(file, key)

                g = file[key]

                x = g["x"][1:500]
                y = g["y"][1:500]

                push!(entropies, spatial_entropy(x, y, i))
            end
        end
    end

    return mean(entropies), entropies
end

H1, Hs1 = mean_entropy(filename1, 1)